# Notebook 05 — Solver Selection and Strategy

## What you will learn
1. How the Newton-Raphson (NR) algorithm works — mathematical derivation
2. Why NR has quadratic convergence near the solution
3. How MAPDL's NROPT variants (FULL, MODI, INIT) differ
4. What automatic time stepping does and when to turn it off
5. How to set convergence tolerances and interpret convergence history
6. The retry/escalation ladder: STABILIZE and arc-length methods
7. How to choose solver type: static, modal, harmonic, buckling

---

## Part 1: Newton-Raphson Algorithm

### The nonlinear problem

After discretization with finite elements, the structural equilibrium problem
becomes a system of nonlinear equations:

$$\mathbf{R}(\mathbf{u}) = \mathbf{F}^{\text{int}}(\mathbf{u}) - \mathbf{F}^{\text{ext}} = \mathbf{0}$$

where:
- $\mathbf{R}$ = residual vector (N)
- $\mathbf{F}^{\text{int}}(\mathbf{u}) = \int_\Omega [B]^T \boldsymbol{\sigma}(\mathbf{u}) \, dV$ (internal forces)
- $\mathbf{F}^{\text{ext}}$ = external force vector (loads)

This is nonlinear because $\boldsymbol{\sigma}$ depends on $\mathbf{u}$ (through the
strain $\boldsymbol{\varepsilon} = [B]\mathbf{u}$ and the constitutive law).

### Newton-Raphson derivation

Taylor-expand $\mathbf{R}(\mathbf{u} + \Delta\mathbf{u})$ about the current solution $\mathbf{u}^k$:

$$\mathbf{R}(\mathbf{u}^k + \Delta\mathbf{u}) \approx \mathbf{R}(\mathbf{u}^k) + \underbrace{\frac{\partial \mathbf{R}}{\partial \mathbf{u}}\bigg|_{\mathbf{u}^k}}_{[K_T(\mathbf{u}^k)]} \Delta\mathbf{u} = \mathbf{0}$$

Solving for $\Delta\mathbf{u}$:

$$\boxed{[K_T(\mathbf{u}^k)] \, \Delta\mathbf{u}^k = -\mathbf{R}(\mathbf{u}^k)}$$

Update: $\mathbf{u}^{k+1} = \mathbf{u}^k + \Delta\mathbf{u}^k$

This is the Newton-Raphson iteration.  $[K_T]$ is called the **tangent stiffness matrix**:

$$[K_T] = \int_\Omega [B]^T [C^{ep}] [B] \, dV + [K_{\sigma}]$$

where $[C^{ep}]$ is the elasto-plastic moduli tensor and $[K_{\sigma}]$ is the
geometric (stress-stiffness) matrix (non-zero only when `NLGEOM ON`).

### Convergence rate

The NR method converges **quadratically** near the solution:

$$\|\mathbf{u}^{k+1} - \mathbf{u}^*\| \leq C \|\mathbf{u}^k - \mathbf{u}^*\|^2$$

This means: if the error is $10^{-3}$ at step $k$, it is $\sim 10^{-6}$ at step $k+1$.
Typically, NR converges in 3–6 iterations for smooth problems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Illustrate quadratic convergence of Newton-Raphson ────────────────────
# Scalar example: solve f(u) = u³ - 2u - 5 = 0 (root ≈ 2.0946)

def f(u):  return u**3 - 2*u - 5
def df(u): return 3*u**2 - 2

u_true = 2.0945514815423265  # true root

# NR iterations starting at u=2
u = 2.0
history_nr = []
for k in range(8):
    r = f(u)
    history_nr.append({'k': k, 'u': u, 'R': r, 'error': abs(u - u_true)})
    du = -r / df(u)
    u += du
    if abs(r) < 1e-14:
        break

# Fixed-point (INIT-like) iterations
K0 = df(2.0)  # constant stiffness
u = 2.0
history_init = []
for k in range(20):
    r = f(u)
    history_init.append({'k': k, 'u': u, 'R': r, 'error': abs(u - u_true)})
    du = -r / K0  # constant tangent
    u += du
    if abs(r) < 1e-14:
        break

# Plot convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: error history
ax = axes[0]
ax.semilogy([h['k'] for h in history_nr],   [h['error'] for h in history_nr],  'b-o', label='FULL NR (quadratic)')
ax.semilogy([h['k'] for h in history_init], [h['error'] for h in history_init], 'r--s', label='INIT (linear)')
ax.set_xlabel('Iteration k'); ax.set_ylabel('|u_k - u*|')
ax.set_title('Convergence Rate Comparison'); ax.legend(); ax.grid(True, alpha=0.3)

# Right: Newton step visualization
ax = axes[1]
u_plot = np.linspace(1.5, 2.5, 200)
ax.plot(u_plot, f(u_plot), 'k-', linewidth=2, label='f(u) = u³ - 2u - 5')
ax.axhline(0, color='gray', linewidth=0.8)
ax.axvline(u_true, color='green', linestyle=':', label=f'True root u* = {u_true:.4f}')

# Show 3 NR steps
u = 2.0
for k in range(3):
    r = f(u); Kt = df(u)
    u_next = u - r/Kt
    ax.plot([u, u_next], [r, 0], 'r--', alpha=0.7)
    ax.plot(u, r, 'ro', markersize=6)
    ax.annotate(f'u_{k}={u:.3f}', (u, r), textcoords='offset points',
                 xytext=(5, 5), fontsize=7, color='red')
    u = u_next

ax.set_xlabel('u'); ax.set_ylabel('f(u)'); ax.set_title('Newton Steps on f(u)=0')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print('NR convergence:')
for h in history_nr:
    print(f'  k={h["k"]}: u={h["u"]:.8f}, error={h["error"]:.2e}')

---

## Part 2: MAPDL NROPT Variants

### FULL Newton-Raphson
- Recomputes and refactors $[K_T]$ at every iteration
- Quadratic convergence — fewest iterations
- **Most expensive per iteration** (requires a full matrix factorization each time)
- **Best for:** strongly nonlinear (large plasticity, large rotation, contact)

### MODI (Modified NR)
- Computes $[K_T]$ once per load step, reuses it for all iterations
- Linear convergence — more iterations needed
- **Cheaper per iteration** (only one factorization per step)
- **Best for:** mildly nonlinear (small plastic deformation, moderate rotation)

### INIT (Initial Stiffness)
- Always uses the elastic stiffness $[K_0]$ from the first load step
- Very slow convergence — but never diverges
- **Best for:** fallback when FULL NR diverges at a load step

### Cost comparison

| Variant | Factorizations per step | Convergence | Typical iterations |
|---------|------------------------|-------------|-------------------|
| FULL    | One per NR iteration   | Quadratic   | 3–6               |
| MODI    | One per load step      | Linear      | 10–20             |
| INIT    | One (ever)             | Sub-linear  | 50+               |

For large models (n_dof > 1M), the factorization cost dominates.  Use MODI
when the material does not change stiffness dramatically within a substep.

In [ ]:
# ── Line search enhancement ────────────────────────────────────────────────
# Line search replaces Δu_k → α Δu_k where α ∈ (0,1] minimizes ‖R(u+αΔu)‖

# 1D illustration: without vs with line search on a problematic function
def f_hard(u):  return np.sign(u) * (np.abs(u)**1.3) - 1.0  # cuspy function
def df_hard(u): return 1.3 * np.abs(u)**0.3

u_star = 1.0  # true root

# Without line search (standard NR)
u = 3.0
hist_no_ls = []
for k in range(20):
    r = f_hard(u); Kt = df_hard(u)
    hist_no_ls.append(abs(u - u_star))
    u -= r / Kt
    if abs(r) < 1e-10: break

# With line search: bisection
u = 3.0
hist_ls = []
for k in range(20):
    r = f_hard(u); Kt = df_hard(u)
    hist_ls.append(abs(u - u_star))
    du = -r / Kt
    # Backtrack: reduce step size until ‖R(u + α du)‖ < ‖R(u)‖
    alpha = 1.0
    for _ in range(20):
        if abs(f_hard(u + alpha * du)) < abs(r):
            break
        alpha *= 0.5
    u += alpha * du
    if abs(r) < 1e-10: break

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(hist_no_ls, 'r-o', label='No line search (may oscillate)')
ax.semilogy(hist_ls,    'b-s', label='With line search (monotone)')
ax.set_xlabel('Iteration'); ax.set_ylabel('|u_k - u*|')
ax.set_title('Effect of Line Search on NR Convergence')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---

## Part 3: Automatic Time Stepping

### What it does

AUTOTS controls the load step size $\Delta t$ (or equivalently, the load fraction
applied per substep).

When the NR iteration fails to converge within NEQIT iterations:
1. AUTOTS **bisects** the load step: $\Delta t \rightarrow \Delta t / 2$
2. Restarts from the last converged state
3. Repeats until either convergence or $\Delta t < \Delta t_{\text{min}}$

When convergence is achieved quickly (< 3 iterations):
- AUTOTS **increases** $\Delta t$ to speed up the next substep

### When to turn AUTOTS OFF

- When you need EXACTLY a certain number of substeps (for time-history output)
- When debugging convergence issues (fixed step helps isolate where divergence occurs)
- For arc-length method (manages its own step size)

### Convergence criterion

MAPDL uses an L2 norm criterion:

$$\text{FNORM} = \frac{\|\mathbf{R}\|_2}{\|\mathbf{F}_{\text{ref}}\|_2} < \varepsilon_F$$

$$\text{UNORM} = \frac{\|\Delta\mathbf{u}\|_2}{\|\mathbf{u}_{\text{ref}}\|_2} < \varepsilon_U$$

Default: $\varepsilon_F = \varepsilon_U = 0.005$ (0.5%).  Both must be satisfied.

In [ ]:
# ── Simulate a convergence history plot (what MAPDL's log looks like) ─────
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n_substeps = 5
threshold  = 0.005

fig, ax = plt.subplots(figsize=(10, 4))
x_offset = 0

for ss in range(n_substeps):
    # Simulate NR convergence: quadratic reduction per iteration
    n_iter = np.random.randint(3, 7)
    r0 = np.random.uniform(0.5, 2.0)  # initial residual
    residuals = [r0]
    for i in range(1, n_iter + 1):
        # Quadratic convergence with some noise
        r_next = residuals[-1]**1.8 * np.random.uniform(0.8, 1.2)
        residuals.append(r_next)
        if r_next < threshold:
            break
    
    x_vals = [x_offset + i for i in range(len(residuals))]
    ax.semilogy(x_vals, residuals, 'b-o', markersize=4)
    ax.axvline(x_vals[-1], color='gray', linestyle=':', alpha=0.5)
    ax.text(x_offset + len(residuals)/2, max(residuals)*1.5,
             f'SS {ss+1}', ha='center', fontsize=8, color='navy')
    x_offset = x_vals[-1] + 1

ax.axhline(threshold, color='red', linestyle='--', label=f'Convergence threshold ε = {threshold}')
ax.set_xlabel('Cumulative iteration number')
ax.set_ylabel('Force residual FNORM')
ax.set_title('Typical NR Convergence History (5 substeps)')
ax.legend(); ax.grid(True, which='both', alpha=0.2)
plt.tight_layout(); plt.show()

print('Reading the convergence plot:')
print('  - Each vertical dashed line = end of a substep')
print('  - Steep drop within a substep = quadratic convergence (good)')
print('  - Flat or oscillating = diverging (increase substeps or check BCs/mesh)')
print('  - Crossing the red line = converged substep')

---

## Part 4: When to Use Each Solver Type

| Analysis type | When to use | Key setting |
|---------------|-------------|-------------|
| **Static**    | Equilibrium under sustained loads | NLGEOM ON/OFF |
| **Modal**     | Natural frequencies and mode shapes | MODOPT,LANB,n_modes |
| **Harmonic**  | Steady-state forced vibration (e.g., vibrating machine) | HARFRQ, DMPRAT |
| **Transient** | Time-domain dynamics (impact, earthquake) | Newmark-β / HHT-α |
| **Buckling**  | Critical buckling load (Euler, shells) | Requires prior static run |

### Buckling analysis workflow

1. **Static solve** with reference load $P_{\text{ref}}$ (e.g., 1 N)
2. **Buckling eigenvalue solve** → extracts $\lambda_i$
3. **Critical load** $P_{\text{cr}} = \lambda_1 \times P_{\text{ref}}$

**Euler column buckling** (both ends pinned, length $L$, cross-section $I$, modulus $E$):
$$P_{\text{cr}} = \frac{\pi^2 E I}{L^2}$$

Use this as a benchmark to verify your MAPDL buckling solve.

In [ ]:
# ── Euler buckling benchmark calculation ──────────────────────────────────
import numpy as np

# Steel column parameters
E   = 200e9   # Young's modulus (Pa)
L   = 1.0     # length (m)
b   = 0.02    # cross-section width (m) — square 20mm × 20mm
h   = 0.02    # cross-section height (m)
I   = b * h**3 / 12   # second moment of area (m⁴)

# Euler critical load
P_cr_both_pinned = np.pi**2 * E * I / L**2
P_cr_one_fixed   = np.pi**2 * E * I / (2*L)**2   # effective length 2L
P_cr_both_fixed  = 4 * np.pi**2 * E * I / L**2   # effective length L/2

print('Euler column buckling loads:')
print(f'  Cross-section: {b*1000:.0f} × {h*1000:.0f} mm, I = {I*1e8:.4f} × 10⁻⁸ m⁴')
print(f'  Both ends pinned: P_cr = {P_cr_both_pinned/1e3:.1f} kN')
print(f'  One end fixed:    P_cr = {P_cr_one_fixed/1e3:.1f} kN')
print(f'  Both ends fixed:  P_cr = {P_cr_both_fixed/1e3:.1f} kN')
print(f'\nFEA should match these within 1-2% with a mesh of 20+ elements.')
print(f'\nTo run a buckling analysis in MAPDL:')
print("  mapdl.run('/SOLU')")
print("  mapdl.antype('STATIC')    # Step 1: static with unit load")
print("  mapdl.f(top_node, 'FY', -1.0)")
print("  mapdl.solve(); mapdl.finish()")
print("  mapdl.antype('BUCKLE')    # Step 2: eigenvalue buckling")
print("  mapdl.bucopt('LANB', 3)  # extract 3 modes")
print("  mapdl.solve(); mapdl.finish()")

---

## Summary

| Decision | Rule |
|----------|------|
| FULL vs MODI | Use FULL unless problem is mildly nonlinear |
| AUTOTS | Always on for nonlinear; off for arc-length |
| Tolerance | Start with 0.5%; tighten to 0.1% if accuracy matters |
| LNSRCH | On for strongly nonlinear; off for linear/modal |
| STABILIZE | Last resort before arc-length |
| Arc-length | Only for snap-through / post-buckling |

**Next:** `06_live_diagnostics.ipynb`